# Thử nghiệm SVR: dự báo điểm Toán mô phỏng

Notebook này dùng `SVR` để dự báo `diem_thi_toan_mo_phong` từ bộ 50.000 hồ sơ mô phỏng. Tệp 10.000 hồ sơ gốc không có nhãn điểm thi, nên không thể dùng riêng tệp đó để huấn luyện bài toán dự báo điểm thi.

> Nhãn là điểm thi do mô phỏng tạo ra, không phải điểm thi thật. Cài thư viện nếu cần: `pip install scikit-learn pandas`.

In [1]:
from pathlib import Path
import numpy as np
import time
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

TARGET = "diem_thi_toan_mo_phong"
DATA_FILE = "dulieuhocsinh_50000_mo_phong.csv"
possible_paths = [Path.cwd() / DATA_FILE, Path.cwd().parent / DATA_FILE]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Không tìm thấy {DATA_FILE}; mở notebook từ thư mục dự án hoặc analys/.")
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()
print(f"Tệp: {DATA_PATH.resolve()}")
print(f"Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print("Số dòng theo phân vùng:")
print(df["tap_du_lieu"].value_counts().sort_index().to_string())

Tệp: D:\Nam4\MLAI\dulieuhocsinh_50000_mo_phong.csv
Kích thước: 50,000 dòng × 131 cột
Số dòng theo phân vùng:
tap_du_lieu
test          10000
train         30000
validation    10000


## SVR là gì?

SVR (Support Vector Regression) là biến thể hồi quy của Support Vector Machine. SVR tìm hàm dự đoán sao cho sai số của phần lớn điểm nằm trong biên cho phép `epsilon`. Các quan sát ảnh hưởng đến nghiệm được gọi là support vectors.

Notebook dùng kernel RBF để mô hình hóa quan hệ phi tuyến. SVR nhạy với thang đo, nên dùng `StandardScaler` trong `Pipeline`; scaler chỉ khớp trên train. Kernel RBF tốn thời gian khi số dòng tăng. Để notebook chạy nhanh, ví dụ lấy mẫu 300 nhóm (1.500 dòng) từ train, giữ đủ 5 biến thể mỗi nhóm. Validation và test dùng đủ dữ liệu; CatBoost cũng được huấn luyện trên đúng mẫu này để so sánh cùng quy mô.

In [2]:
group_sets = {name: set(part["nhom_ho_so_goc"].unique()) for name, part in df.groupby("tap_du_lieu")}
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = group_sets[left] & group_sets[right]
    print(f"Nhóm trùng giữa {left} và {right}: {len(overlap)}")
    assert not overlap, f"Rò rỉ nhóm hồ sơ giữa {left} và {right}"
train_df_full = df.loc[df["tap_du_lieu"] == "train"].copy()
valid_df = df.loc[df["tap_du_lieu"] == "validation"].copy()
test_df = df.loc[df["tap_du_lieu"] == "test"].copy()
selected_groups = train_df_full["nhom_ho_so_goc"].drop_duplicates().sample(n=300, random_state=42)
train_df = train_df_full.loc[train_df_full["nhom_ho_so_goc"].isin(selected_groups)].copy()
print(f"Số nhóm nguồn: train đầy đủ={train_df_full['nhom_ho_so_goc'].nunique():,}, mẫu train={train_df['nhom_ho_so_goc'].nunique():,}, validation={valid_df['nhom_ho_so_goc'].nunique():,}, test={test_df['nhom_ho_so_goc'].nunique():,}")
print(f"Số dòng train lấy mẫu: {len(train_df):,} / {len(train_df_full):,}")

Nhóm trùng giữa train và validation: 0
Nhóm trùng giữa train và test: 0
Nhóm trùng giữa validation và test: 0
Số nhóm nguồn: train đầy đủ=6,000, mẫu train=300, validation=2,000, test=2,000
Số dòng train lấy mẫu: 1,500 / 30,000


## Nạp dữ liệu và kiểm tra phân vùng

Dữ liệu đã có các phân vùng train/validation/test. Kiểm tra rằng `nhom_ho_so_goc` không bị chia qua các phân vùng, vì các biến thể cùng hồ sơ có quan hệ với nhau.

In [3]:
academic_cols = [c for c in df.columns if c.endswith(tuple(f"_lop{g}" for g in range(6, 10)))]
feature_cols = academic_cols + ["gio_tu_hoc_tuan", "ti_le_chuyen_can", "diem_thi_thu_toan"]
X_train, y_train = train_df[feature_cols], train_df["diem_thi_toan_mo_phong"]
X_valid, y_valid = valid_df[feature_cols], valid_df["diem_thi_toan_mo_phong"]
X_test, y_test = test_df[feature_cols], test_df["diem_thi_toan_mo_phong"]
print(f"Số đặc trưng: {len(feature_cols)}")
print(f"Số dòng đánh giá: validation={len(valid_df):,}, test={len(test_df):,}")
from catboost import CatBoostRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
catboost_model = CatBoostRegressor(iterations=1000, depth=8, learning_rate=0.05,
    loss_function="RMSE", eval_metric="RMSE", random_seed=42, verbose=False, allow_writing_files=False)
catboost_model.fit(X_train, y_train, eval_set=(X_valid, y_valid), early_stopping_rounds=100, verbose=False)
svr_model = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale", tol=0.01, max_iter=5000))
start_time = time.perf_counter()
svr_model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - start_time
print("Đã huấn luyện CatBoost và SVR trên cùng 1.500 dòng train.")
print(f"SVR fit time: {fit_seconds:.1f} giây")
print(f"SVR fit_status: {svr_model.named_steps['svr'].fit_status_} (0=hội tụ, 1=chạm giới hạn vòng lặp)")

Số đặc trưng: 55
Số dòng đánh giá: validation=10,000, test=10,000
Đã huấn luyện CatBoost và SVR trên cùng 1.500 dòng train.
SVR fit time: 0.2 giây
SVR fit_status: 1 (0=hội tụ, 1=chạm giới hạn vòng lặp)


## Chuẩn bị đặc trưng và huấn luyện

Dùng cùng 55 đặc trưng như notebook CatBoost: điểm môn học theo lớp, giờ tự học, chuyên cần và điểm thi thử Toán. Không dùng định danh, nhóm hồ sơ, nhãn đích hoặc thông tin tuyển sinh tương lai.

SVR dùng kernel RBF với `C=10`, `epsilon=0.1`, `gamma="scale"`, `tol=0.01`. Tham số được cố định để minh họa; chưa dò tìm tối ưu.

In [4]:
def calculate_metrics(actual, predicted):
    return {"MAE": mean_absolute_error(actual, predicted), "RMSE": np.sqrt(mean_squared_error(actual, predicted)), "R²": r2_score(actual, predicted)}
rows = []
for split_name, part, X, y in [("validation", valid_df, X_valid, y_valid), ("test", test_df, X_test, y_test)]:
    for model_name, fitted_model in [("SVR (RBF)", svr_model), ("CatBoost (1.500 dòng)", catboost_model)]:
        rows.append({"Tập": split_name, "Mô hình": model_name, **calculate_metrics(y, fitted_model.predict(X))})
    rows.append({"Tập": split_name, "Mô hình": "Mốc: toan_lop9", **calculate_metrics(y, part["toan_lop9"])})
results = pd.DataFrame(rows)
print(results.to_string(index=False, formatters={"MAE": "{:.3f}".format, "RMSE": "{:.3f}".format, "R²": "{:.3f}".format}))

       Tập               Mô hình   MAE  RMSE    R²
validation             SVR (RBF) 0.914 1.149 0.808
validation CatBoost (1.500 dòng) 0.788 0.983 0.859
validation        Mốc: toan_lop9 1.021 1.303 0.753
      test             SVR (RBF) 0.924 1.158 0.802
      test CatBoost (1.500 dòng) 0.810 1.009 0.849
      test        Mốc: toan_lop9 1.065 1.352 0.730


## Đánh giá và so sánh

MAE và RMSE càng thấp càng tốt; R² càng cao càng tốt. So sánh SVR, CatBoost được huấn luyện trên cùng 1.500 dòng train và mốc `toan_lop9`. Cả ba được đánh giá trên cùng validation/test.

Notebook CatBoost trước đó dùng đủ 30.000 dòng train. Không so sánh trực tiếp với CatBoost đầy đủ nếu chưa cân bằng số dòng huấn luyện.

In [5]:
fitted_svr = svr_model.named_steps["svr"]
print(f"Số support vectors: {len(fitted_svr.support_):,} / {len(X_train):,} dòng train")

Số support vectors: 1,390 / 1,500 dòng train


## Support vectors

SVR lưu các điểm train đóng góp vào hàm dự đoán dưới dạng support vectors. Số lượng nhiều có thể làm dự đoán chậm hơn; huấn luyện SVR kernel cũng thường tốn kém hơn khi số dòng tăng.

In [6]:
preview = pd.DataFrame({
    "Điểm Toán lớp 9 (mốc)": test_df["toan_lop9"].head(5).to_numpy(),
    "Điểm thi mô phỏng": y_test.head(5).to_numpy(),
    "Dự đoán SVR": svr_model.predict(X_test.head(5)),
})
print(preview.round(2).to_string(index=False))

 Điểm Toán lớp 9 (mốc)  Điểm thi mô phỏng  Dự đoán SVR
                   2.0               1.04         3.20
                   8.0               5.60         4.63
                   4.0               1.27         3.62
                   3.0               6.28         3.98
                   5.0               4.86         5.91


## Diễn giải kết quả

Xem bảng validation/test để so sánh cấu hình SVR này với CatBoost được huấn luyện trên cùng mẫu và mốc `toan_lop9`. Vì chỉ lấy 1.500 dòng train để minh họa tốc độ, điểm số có thể khác so với CatBoost dùng đủ 30.000 dòng. Không thể kết luận SVR nói chung tốt hay kém hơn từ một cấu hình; cần dò tham số theo quy trình thống nhất.

Các nhãn là điểm thi mô phỏng. Kết quả không chứng minh độ chính xác với học sinh thực tế.